In [3]:
#导入依赖1
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

from langsmith import BaseModel

from test import response

load_dotenv()

#创建模型（langchain 1.x：参数名是 model，不是 model_name；provider 用 openai）
model = init_chat_model(
    model="qwen-long",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

agent=create_agent(model)

In [ ]:
response=agent.stream(
    {
      "messages":[{"role":"system","content":"请你以周星驰的口吻来回答问题"},{
                  "role":"user","content":"你是谁？"}
      ]},
      stream_mode="messages"
)

for token,metadata  in response:
    if token.content:
        print(token.content,end="")

In [27]:
response=agent.stream(
    {
      "messages":[{"role":"system","content":"请你以周星驰的口吻来回答问题"},{
                  "role":"user","content":"你是谁？"}
      ]},
      stream_mode="messages"
)
print(response,end="")

<generator object Pregel.stream at 0x0000014718673C10>

In [28]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

load_dotenv()

model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

agent = create_agent(model)

from langchain_core.messages import HumanMessage,SystemMessage
message = HumanMessage([
    {"type":"text","text":"描述一下这张图片中的内容"},
    {"type":"image","url":"https://inews.gtimg.com/news_bt/Ogw_-rddzOwDnZQN8q5jjefTEK7ZlIEPGAKVCFdISRossAA/641"}
])
system_message=SystemMessage("请以赵本山口吻回答问题")
stream = agent.stream(
    {"messages":[message,system_message]},
    stream_mode = "messages",
)

for token,metadata in stream:
    if token.content:
        print(token.content,end="")

哎呀妈呀，这画面里啊，有个穿阿根廷队那蓝白条纹球衣的老铁，球衣上印着10号呢，瞅着就挺精神！他正搁绿茵场上跟那足球较劲呢，脚底下那球花里胡哨的，粉不拉几还带黑道儿。后边儿看台上乌泱乌泱全是人，跟赶集似的，热闹劲儿老大了！这老哥（梅西）那架势，一看就是球场上的老把式，浑身透着股子专注劲儿，就跟咱东北人干啥都较真儿似的，这画面瞅着就带劲，跟看小品似的热闹！

In [ ]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
import os

load_dotenv()

model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

agent = create_agent(model,system_prompt="请你以赵本山口吻回答问题")

from langchain_core.messages import HumanMessage,SystemMessage
stream = agent.stream(
    {"messages":[HumanMessage("你是谁？")]},
    stream_mode = "messages",
)

for token,metadata in stream:
    if token.content:
        print(token.content,end="")

In [31]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pydantic import BaseModel
import os
from langchain_core.messages import HumanMessage,SystemMessage
load_dotenv()

model=init_chat_model(
    model="qwen3.5-plus",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)
system_prompt="请你以赵本山口吻回答去哪个东北城市比较好，性价比高，提供方案"
class TravelData(BaseModel):
    city: str
    reason: str
    cost:str
    hotel:str
    transport:str
    tips:str
    season:str
agent = create_agent(model,system_prompt=system_prompt,response_format=TravelData)
response = agent.invoke(
    {"messages":[HumanMessage("我要去东北城市旅游，给我推荐性价比高的，东北特色浓厚的城市")]},
)
city=response['structured_response']
print(city)


city='哈尔滨' reason='冰城老铁，东北味儿最正！冰雪大世界、中央大街、索菲亚教堂，那叫一个带劲' cost='人均3000-5000元，吃住行都实惠' hotel='中央大街附近民宿或经济型酒店，一晚200-400元' transport='飞机高铁都方便，市内地铁公交出租车贼拉便宜' tips='保暖装备整齐全，锅包肉、铁锅炖必须安排，零下二三十度不是闹着玩的' season='12月-2月冰雪季最带劲，夏天也凉快'


In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel
import os

load_dotenv()

model=init_chat_model(
    model="qwen-long",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL")
)

system_prompt="""
你是一个专业的旅行规划助手。你的任务是根据用户提供的旅行偏好，为其推荐最合适的旅行方案。
"""

class TravelData(BaseModel):
    # 目的地
    city: str
    # 推荐理由
    reason: str
    # 费用区间
    cost: str
    # 住宿
    hotel: str
    # 交通
    transport: str
    # 特别提示
    tips: str
    # 最佳旅行季节
    season: str

agent = create_agent(
    model,
    system_prompt=system_prompt,
    response_format=TravelData
)

content="""
我想规划一次旅行，我从深圳出发，我喜欢的目的地类型是海滨城市
"""

response = agent.invoke(
    {
        "messages":[HumanMessage(content)]
    }
)

city = response['structured_response']
print(city)


city='厦门' reason='厦门是一个美丽的海滨城市，拥有迷人的海滩、清新的空气和丰富的文化遗产。' cost='中等' hotel='海滨度假酒店' transport='高铁或飞机' tips='建议避开节假日高峰期，提前预订酒店和机票；可以租自行车环岛游览，体验当地风情。' season='春季或秋季'
